# Grace in the Gap — technical verification notebook

**Scripture for the seconds when AI asks us to wait.** Grace in the Gap is a Claude Code plugin that turns an AI wait state into an optional five-second breath, a pre-authored reflection, and an attributed passage of Scripture—then returns the developer to the completed work.

This is a judge-facing verification companion to the production TypeScript repository. It is deliberately self-contained and runs without credentials. It proves the privacy boundary, reproduces the production weighted selector exactly across all 108 evaluation scenarios, and exercises strict rejection of malformed or cross-wired provider decisions. The final section is the real Gloo + YouVersion path and runs only when private Kaggle Secrets are present.

| Evidence | What this notebook verifies |
|---|---|
| Privacy | Raw prompts, code, paths, and transcripts cannot enter the provider payload |
| Grounding | Gloo may choose only IDs from the approved local catalog |
| Safety | Strict shape and catalog cross-validation reject generated prose and mismatched IDs |
| Reliability | 108/108 task × time-window × wait-length scenarios resolve to approved content |
| Attribution | The live path requires YouVersion passage text plus Bible metadata before rendering |


## 1. Pinned production evidence

The production implementation is the source of truth. This notebook is pinned to the first audited product commit and carries a canonical snapshot of that commit's catalog so the verification remains runnable even when Kaggle internet access is disabled.


In [1]:
from collections import Counter
from copy import deepcopy
from hashlib import sha256
from itertools import product
import base64
import json
import math
import os
import re

PROJECT_URL = 'https://github.com/Marc-Dvci/Grace-in-the-Gap'
PRODUCTION_COMMIT = 'd897c7f38c2f1a00a60879d1c2e028483d135c46'
CATALOG_RELEASE = '2026.07-demo.1'
EXPECTED_CATALOG_SHA256 = '77fba58be8cd1cc51ef921b8f008757a8556a5d5a369b763e164a8f560600a04'

print(f'Production commit: {PRODUCTION_COMMIT[:7]}')
print(f'Source: {PROJECT_URL}/tree/{PRODUCTION_COMMIT}')


Production commit: d897c7f
Source: https://github.com/Marc-Dvci/Grace-in-the-Gap/tree/d897c7f38c2f1a00a60879d1c2e028483d135c46


In [2]:
# Canonical snapshot of content/catalog.json at PRODUCTION_COMMIT.
CATALOG = {
  "release": "2026.07-demo.1",
  "review_notice": "Conservative demo copy. A qualified theology/editorial reviewer must approve this release before public launch.",
  "offline_passages_notice": "Public-domain World English Bible verses, used only as an offline fallback when the live YouVersion API is unreachable.",
  "profiles": [
    {"id": "quiet-trust", "task_types": ["generation", "analysis", "unknown"], "tone": "calm", "snippet_id": "pause-and-release", "passage_hint": "PSA.46.10", "weight": 1},
    {"id": "purposeful-work", "task_types": ["implementation", "refactor"], "tone": "steady", "snippet_id": "work-with-purpose", "passage_hint": "COL.3.23", "weight": 1},
    {"id": "wisdom-in-debugging", "task_types": ["debugging"], "tone": "reflective", "snippet_id": "make-room-for-wisdom", "passage_hint": "JAS.1.5", "weight": 1},
    {"id": "patient-in-testing", "task_types": ["testing"], "tone": "encouraging", "snippet_id": "hope-while-it-runs", "passage_hint": "ROM.12.12", "weight": 1},
    {"id": "committed-plans", "task_types": ["planning"], "tone": "steady", "snippet_id": "hold-the-plan-lightly", "passage_hint": "PRO.16.3", "weight": 1},
    {"id": "thoughtful-review", "task_types": ["review"], "tone": "reflective", "snippet_id": "notice-what-is-good", "passage_hint": "PHP.4.8", "weight": 1},
    {"id": "rest-for-late-work", "task_types": ["generation", "implementation", "debugging", "testing", "analysis", "refactor", "planning", "review", "unknown"], "tone": "calm", "snippet_id": "you-can-rest", "passage_hint": "MAT.11.28", "time_windows": ["late-evening"], "weight": 1.25},
    {"id": "care-in-uncertainty", "task_types": ["debugging", "analysis", "unknown"], "tone": "encouraging", "snippet_id": "set-down-the-worry", "passage_hint": "1PE.5.7", "weight": 0.9}
  ],
  "snippets": [
    {"id": "pause-and-release", "locale": "en-US", "text": "One slow breath. The result can arrive without carrying all your attention.", "status": "approved-for-demo"},
    {"id": "work-with-purpose", "locale": "en-US", "text": "Let the build run. Return to the purpose beneath the task.", "status": "approved-for-demo"},
    {"id": "make-room-for-wisdom", "locale": "en-US", "text": "The bug can wait five seconds. Make room for wisdom before the next attempt.", "status": "approved-for-demo"},
    {"id": "hope-while-it-runs", "locale": "en-US", "text": "While the tests run, loosen your shoulders and hold on to hope.", "status": "approved-for-demo"},
    {"id": "hold-the-plan-lightly", "locale": "en-US", "text": "Commit the work. Hold the outcome with open hands.", "status": "approved-for-demo"},
    {"id": "notice-what-is-good", "locale": "en-US", "text": "Before finding the flaw, take a moment to notice what is true and good.", "status": "approved-for-demo"},
    {"id": "you-can-rest", "locale": "en-US", "text": "This task is not a measure of your worth. Receive a moment of rest.", "status": "approved-for-demo"},
    {"id": "set-down-the-worry", "locale": "en-US", "text": "Name the worry, then set it down for the length of one breath.", "status": "approved-for-demo"}
  ],
  "offline_passages": [
    {"usfm": "PSA.46.10", "reference": "Psalm 46:10", "text": "Be still, and know that I am God. I will be exalted among the nations. I will be exalted in the earth.", "version_id": "WEB", "version_name": "World English Bible", "copyright": "World English Bible — Public Domain", "locale": "en-US"},
    {"usfm": "COL.3.23", "reference": "Colossians 3:23", "text": "And whatever you do, work heartily, as for the Lord, and not for men,", "version_id": "WEB", "version_name": "World English Bible", "copyright": "World English Bible — Public Domain", "locale": "en-US"},
    {"usfm": "JAS.1.5", "reference": "James 1:5", "text": "But if any of you lacks wisdom, let him ask of God, who gives to all liberally and without reproach; and it will be given to him.", "version_id": "WEB", "version_name": "World English Bible", "copyright": "World English Bible — Public Domain", "locale": "en-US"},
    {"usfm": "ROM.12.12", "reference": "Romans 12:12", "text": "rejoicing in hope, enduring in troubles, continuing steadfastly in prayer,", "version_id": "WEB", "version_name": "World English Bible", "copyright": "World English Bible — Public Domain", "locale": "en-US"},
    {"usfm": "PRO.16.3", "reference": "Proverbs 16:3", "text": "Commit your deeds to Yahweh, and your plans shall succeed.", "version_id": "WEB", "version_name": "World English Bible", "copyright": "World English Bible — Public Domain", "locale": "en-US"},
    {"usfm": "PHP.4.8", "reference": "Philippians 4:8", "text": "Finally, brothers, whatever things are true, whatever things are honorable, whatever things are just, whatever things are pure, whatever things are lovely, whatever things are of good report: if there is any virtue and if there is anything worthy of praise, think about these things.", "version_id": "WEB", "version_name": "World English Bible", "copyright": "World English Bible — Public Domain", "locale": "en-US"},
    {"usfm": "MAT.11.28", "reference": "Matthew 11:28", "text": "Come to me, all you who labor and are heavily burdened, and I will give you rest.", "version_id": "WEB", "version_name": "World English Bible", "copyright": "World English Bible — Public Domain", "locale": "en-US"},
    {"usfm": "1PE.5.7", "reference": "1 Peter 5:7", "text": "casting all your worries on him, because he cares for you.", "version_id": "WEB", "version_name": "World English Bible", "copyright": "World English Bible — Public Domain", "locale": "en-US"}
  ]
}


In [3]:
def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
    return sha256(payload.encode('utf-8')).hexdigest()

assert CATALOG['release'] == CATALOG_RELEASE
assert canonical_hash(CATALOG) == EXPECTED_CATALOG_SHA256

profiles = CATALOG['profiles']
snippets = {item['id']: item for item in CATALOG['snippets']}
passages = {item['usfm']: item for item in CATALOG['offline_passages']}
assert len({item['id'] for item in profiles}) == len(profiles)
assert all(profile['snippet_id'] in snippets for profile in profiles)
assert all(profile['passage_hint'] in passages for profile in profiles)
assert all(snippet['status'] == 'approved-for-demo' for snippet in snippets.values())

print(f'Catalog release: {CATALOG_RELEASE}')
print(f'Canonical SHA-256: {EXPECTED_CATALOG_SHA256}')
print(f'{len(profiles)} profiles · {len(snippets)} approved snippets · {len(passages)} fallback passages')


Catalog release: 2026.07-demo.1
Canonical SHA-256: 77fba58be8cd1cc51ef921b8f008757a8556a5d5a369b763e164a8f560600a04
8 profiles · 8 approved snippets · 8 fallback passages


## 2. Exact privacy and selection boundary

The provider never receives the raw prompt, code, filename, path, transcript, or session hash. It receives only six allow-listed fields. The on-device fallback below is a line-for-line behavioral port of `src/selection/local-selector.ts`: FNV-1a hashing, weight-expanded candidates, and the same duration rule.


In [4]:
TASK_TYPES = ['generation', 'implementation', 'debugging', 'testing', 'analysis', 'refactor', 'planning', 'review', 'unknown']
TIME_WINDOWS = ['morning', 'afternoon', 'evening', 'late-evening']
TONES = {'calm', 'steady', 'encouraging', 'reflective'}
DECISION_KEYS = {
    'momentProfileId', 'reflectionSnippetId', 'passageHint', 'durationSeconds',
    'tone', 'confidence', 'fallbackVotd', 'needsAuth'
}
USFM_PATTERN = re.compile(r'^[1-3]?[A-Z]{2,3}\.[0-9]+\.[0-9]+(?:-[0-9]+)?$')

def duration_bucket(seconds):
    if seconds < 8: return 'under-8'
    if seconds <= 15: return '8-15'
    if seconds <= 30: return '16-30'
    return 'over-30'

def build_event(task_type, estimated_wait_seconds, time_window, session_seed, surface='demo'):
    assert task_type in TASK_TYPES and time_window in TIME_WINDOWS
    assert isinstance(estimated_wait_seconds, int) and 0 <= estimated_wait_seconds <= 600
    return {
        'surface': surface,
        'taskType': task_type,
        'estimatedWaitSeconds': estimated_wait_seconds,
        'durationBucket': duration_bucket(estimated_wait_seconds),
        'locale': 'en-US',
        'timeWindow': time_window,
        'sessionHash': sha256(session_seed.encode()).hexdigest()[:24],
        'contextMode': 'private',
    }

def provider_payload(event):
    # Exact allow-list used by the production Gloo adapter.
    return {key: event[key] for key in (
        'surface', 'taskType', 'durationBucket', 'locale', 'timeWindow', 'contextMode'
    )}

def candidates_for(event):
    return [profile for profile in profiles
            if event['taskType'] in profile['task_types']
            and ('time_windows' not in profile or event['timeWindow'] in profile['time_windows'])]

def stable_number(seed):
    # JavaScript FNV-1a with Math.imul(...), returned as uint32.
    value = 2166136261
    for character in seed:
        value ^= ord(character)
        value = (value * 16777619) & 0xFFFFFFFF
    return value

def js_round_positive(value):
    return math.floor(value + 0.5)

def select_locally(event, candidates):
    if not candidates:
        raise ValueError('No eligible moment profiles')
    weighted = []
    for profile in candidates:
        weighted.extend([profile] * max(1, js_round_positive(profile['weight'] * 4)))
    seed = f"{event['sessionHash']}:{event['taskType']}:{event['timeWindow']}"
    selected = weighted[stable_number(seed) % len(weighted)]
    return {
        'momentProfileId': selected['id'],
        'reflectionSnippetId': selected['snippet_id'],
        'passageHint': selected['passage_hint'],
        'durationSeconds': 8 if event['estimatedWaitSeconds'] >= 16 else 5,
        'tone': selected['tone'],
        'confidence': 0.82,
        'fallbackVotd': False,
        'needsAuth': False,
    }


In [5]:
def validate_provider_decision(decision, candidates):
    if type(decision) is not dict or set(decision) != DECISION_KEYS:
        return False, 'strict schema mismatch'
    if not all(type(decision[key]) is str for key in ('momentProfileId', 'reflectionSnippetId', 'passageHint', 'tone')):
        return False, 'identifier type mismatch'
    if type(decision['durationSeconds']) is not int or not 3 <= decision['durationSeconds'] <= 20:
        return False, 'duration out of bounds'
    confidence = decision['confidence']
    if isinstance(confidence, bool) or not isinstance(confidence, (int, float)) or not 0 <= confidence <= 1:
        return False, 'confidence out of bounds'
    if type(decision['fallbackVotd']) is not bool or type(decision['needsAuth']) is not bool:
        return False, 'boolean type mismatch'
    if decision['tone'] not in TONES or not USFM_PATTERN.fullmatch(decision['passageHint']):
        return False, 'tone or USFM format mismatch'
    if confidence < 0.55 or decision['needsAuth'] or decision['fallbackVotd']:
        return False, 'provider policy rejected'
    profile = next((item for item in candidates if item['id'] == decision['momentProfileId']), None)
    if not profile:
        return False, 'unknown profile'
    exact_match = (
        profile['snippet_id'] == decision['reflectionSnippetId']
        and profile['passage_hint'] == decision['passageHint']
        and profile['tone'] == decision['tone']
    )
    return (True, 'accepted') if exact_match else (False, 'catalog cross-check rejected')

raw_prompt = 'Diagnose the failing integration test in C:/private/client-repo and inspect customer_code.ts'
event = build_event('debugging', 12, 'afternoon', 'kaggle-judge-session')
safe_payload = provider_payload(event)
decision = select_locally(event, candidates_for(event))
accepted, reason = validate_provider_decision(decision, candidates_for(event))

serialized = json.dumps(safe_payload)
assert raw_prompt not in serialized and 'client-repo' not in serialized and 'customer_code.ts' not in serialized
assert set(safe_payload) == {'surface', 'taskType', 'durationBucket', 'locale', 'timeWindow', 'contextMode'}
assert accepted, reason

snippet = snippets[decision['reflectionSnippetId']]['text']
reference = passages[decision['passageHint']]['reference']
print('Provider-safe payload:')
print(json.dumps(safe_payload, indent=2))
print(f'\nSelected: {decision["momentProfileId"]} · {reference} · {decision["durationSeconds"]}s')
print(f'Reflection: {snippet}')
print('Privacy assertion: PASS — raw prompt, code, and paths are absent')


Provider-safe payload:
{
  "surface": "demo",
  "taskType": "debugging",
  "durationBucket": "8-15",
  "locale": "en-US",
  "timeWindow": "afternoon",
  "contextMode": "private"
}

Selected: wisdom-in-debugging · James 1:5 · 5s
Reflection: The bug can wait five seconds. Make room for wisdom before the next attempt.
Privacy assertion: PASS — raw prompt, code, and paths are absent


## 3. Production-parity evaluation: 108 scenarios

This is the same Cartesian evaluation used by `src/evaluation/run.ts`: nine task types × four time windows × three wait lengths. The expected distribution is pinned to the audited TypeScript result, so an apparently successful but behaviorally different Python port fails this cell.


In [6]:
EXPECTED_DISTRIBUTION = {
    'PSA.46.10': 19, 'MAT.11.28': 14, 'COL.3.23': 19, '1PE.5.7': 19,
    'JAS.1.5': 5, 'ROM.12.12': 10, 'PRO.16.3': 10, 'PHP.4.8': 12,
}
waits = [8, 12, 20]
distribution = Counter()
failures = []
count = 0

for task_type, time_window, estimated_wait_seconds in product(TASK_TYPES, TIME_WINDOWS, waits):
    count += 1
    try:
        # Production evaluation intentionally uses this literal sessionHash.
        scenario = {
            'surface': 'demo', 'taskType': task_type,
            'estimatedWaitSeconds': estimated_wait_seconds,
            'durationBucket': '8-15' if estimated_wait_seconds <= 15 else '16-30',
            'locale': 'en-US', 'timeWindow': time_window,
            'sessionHash': f'eval-session-{count:04d}', 'contextMode': 'private',
        }
        eligible = candidates_for(scenario)
        selected = select_locally(scenario, eligible)
        valid, why = validate_provider_decision(selected, eligible)
        assert valid, why
        assert snippets[selected['reflectionSnippetId']]['status'] == 'approved-for-demo'
        assert selected['passageHint'] in passages
        distribution[selected['passageHint']] += 1
    except Exception as exc:
        failures.append((task_type, time_window, estimated_wait_seconds, str(exc)))

result = {
    'scenarios': count, 'passed': count - len(failures), 'failures': len(failures),
    'schemaComplianceRate': (count - len(failures)) / count,
    'uniquePassages': len(distribution), 'distribution': dict(distribution),
}
assert not failures
assert dict(distribution) == EXPECTED_DISTRIBUTION, 'Python port drifted from production TypeScript'
print(json.dumps(result, indent=2))
print('\nPARITY CHECK: PASS — exact TypeScript distribution reproduced')


{
  "scenarios": 108,
  "passed": 108,
  "failures": 0,
  "schemaComplianceRate": 1.0,
  "uniquePassages": 8,
  "distribution": {
    "PSA.46.10": 19,
    "MAT.11.28": 14,
    "COL.3.23": 19,
    "1PE.5.7": 19,
    "JAS.1.5": 5,
    "ROM.12.12": 10,
    "PRO.16.3": 10,
    "PHP.4.8": 12
  }
}

PARITY CHECK: PASS — exact TypeScript distribution reproduced


## 4. Adversarial provider boundary

Gloo is a selector, not a devotional writer. The service treats every model response as untrusted: strict keys and types must pass, then the profile, reflection, passage, and tone must match one approved catalog row exactly.


In [7]:
eligible = candidates_for(event)
valid_choice = deepcopy(decision)
extra_prose = {**decision, 'reflection': 'Model-authored devotional text'}
cross_wired = {**decision, 'passageHint': 'ROM.12.12'}
low_confidence = {**decision, 'confidence': 0.10}
auth_required = {**decision, 'needsAuth': True}

cases = {
    'approved exact choice': valid_choice,
    'extra generated prose': extra_prose,
    'cross-wired passage ID': cross_wired,
    'low confidence': low_confidence,
    'authorization requested': auth_required,
}
outcomes = {}
for name, candidate in cases.items():
    outcomes[name] = validate_provider_decision(candidate, eligible)
    accepted_case, reason_case = outcomes[name]
    print(f'{name:27} {"ACCEPTED" if accepted_case else "REJECTED":8} · {reason_case}')

assert outcomes['approved exact choice'][0] is True
assert all(outcomes[name][0] is False for name in outcomes if name != 'approved exact choice')
print('\nADVERSARIAL CHECK: PASS — unsafe outputs never reach the user')


approved exact choice       ACCEPTED · accepted
extra generated prose       REJECTED · strict schema mismatch
cross-wired passage ID      REJECTED · catalog cross-check rejected
low confidence              REJECTED · provider policy rejected
authorization requested     REJECTED · provider policy rejected

ADVERSARIAL CHECK: PASS — unsafe outputs never reach the user


## 5. Real Gloo + YouVersion integration

This cell uses the actual production endpoints and request shapes. Add `GLOO_CLIENT_ID`, `GLOO_CLIENT_SECRET`, and `YVP_APP_KEY` as private Kaggle Secrets, then run the notebook. Secrets and access tokens are never printed. If credentials are absent, the cell exits cleanly instead of fabricating a result.

The configured Bible version ID is intentionally treated as an opaque YouVersion identifier. Its version name and required copyright are resolved from `/v1/bibles/{id}` at runtime—nothing is guessed or hard-coded.


In [8]:
def get_credentials():
    names = ('GLOO_CLIENT_ID', 'GLOO_CLIENT_SECRET', 'YVP_APP_KEY')
    credentials = {name: os.environ.get(name, '') for name in names}
    if not all(credentials.values()):
        try:
            from kaggle_secrets import UserSecretsClient
            client = UserSecretsClient()
            for name in names:
                if not credentials[name]:
                    try:
                        credentials[name] = client.get_secret(name)
                    except Exception:
                        pass
        except Exception:
            pass
    return credentials

def build_instructions(candidates):
    allowed = [{
        'momentProfileId': profile['id'],
        'reflectionSnippetId': profile['snippet_id'],
        'passageHint': profile['passage_hint'],
        'tone': profile['tone'],
    } for profile in candidates]
    return '\n'.join([
        'You are a structured selector for Grace in the Gap.',
        'Return one valid JSON object only: no Markdown and no user-facing prose.',
        'Choose one allowed candidate without changing any candidate field.',
        'Use durationSeconds 5 or 8, confidence from 0 to 1, fallbackVotd false, and needsAuth false.',
        f'Allowed candidates: {json.dumps(allowed, separators=(",", ":"))}',
        'Required camelCase keys: momentProfileId, reflectionSnippetId, passageHint, durationSeconds, tone, confidence, fallbackVotd, needsAuth.',
    ])

def extract_gloo_text(body):
    choices = body.get('choices')
    if isinstance(choices, list) and choices:
        content = choices[0].get('message', {}).get('content')
        if isinstance(content, str):
            return content.strip()
    for item in body.get('output', []):
        if item.get('type') != 'message':
            continue
        for part in item.get('content', []):
            if isinstance(part.get('text'), str):
                return part['text'].strip()
    raise ValueError('Gloo response did not contain assistant JSON text')

def run_live_card(credentials):
    import requests
    gloo_base = os.environ.get('GLOO_BASE_URL', 'https://platform.ai.gloo.com').rstrip('/')
    youversion_base = os.environ.get('YOUVERSION_BASE_URL', 'https://api.youversion.com').rstrip('/')
    model = os.environ.get('GLOO_MODEL', 'gloo-openai-gpt-5-mini')
    version_id = os.environ.get('GRACE_BIBLE_VERSION_ID', '3034')

    basic = base64.b64encode(
        f"{credentials['GLOO_CLIENT_ID']}:{credentials['GLOO_CLIENT_SECRET']}".encode()
    ).decode()
    token_response = requests.post(
        f'{gloo_base}/oauth2/token',
        headers={'Authorization': f'Basic {basic}', 'Content-Type': 'application/x-www-form-urlencoded'},
        data={'grant_type': 'client_credentials', 'scope': 'api/access'}, timeout=15,
    )
    token_response.raise_for_status()
    token = token_response.json()['access_token']

    live_event = build_event('debugging', 12, 'afternoon', 'kaggle-live-session')
    eligible = candidates_for(live_event)
    gloo_response = requests.post(
        f'{gloo_base}/ai/v1/responses',
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
        json={
            'model': model, 'instructions': build_instructions(eligible),
            'input': [{'role': 'user', 'content': json.dumps(provider_payload(live_event))}],
        }, timeout=30,
    )
    gloo_response.raise_for_status()
    provider_decision = json.loads(extract_gloo_text(gloo_response.json()))
    accepted, rejection_reason = validate_provider_decision(provider_decision, eligible)
    degraded = not accepted
    selected = provider_decision if accepted else select_locally(live_event, eligible)

    headers = {
        'X-YVP-App-Key': credentials['YVP_APP_KEY'],
        'Accept-Language': live_event['locale'], 'Accept': 'application/json',
    }
    passage_response = requests.get(
        f"{youversion_base}/v1/bibles/{version_id}/passages/{selected['passageHint']}",
        params={'format': 'text'}, headers=headers, timeout=15,
    )
    metadata_response = requests.get(
        f'{youversion_base}/v1/bibles/{version_id}', headers=headers, timeout=15,
    )
    passage_response.raise_for_status(); metadata_response.raise_for_status()
    passage_body = passage_response.json().get('data', passage_response.json())
    metadata_body = metadata_response.json().get('data', metadata_response.json())
    text = passage_body.get('content') or passage_body.get('text') or passage_body.get('passage_text')
    version = (metadata_body.get('abbreviation') or metadata_body.get('localized_abbreviation')
               or metadata_body.get('title') or metadata_body.get('localized_title'))
    copyright_text = metadata_body.get('copyright') or metadata_body.get('promotional_content')
    if not text or not version or not copyright_text:
        raise ValueError('YouVersion response lacked text, version, or required copyright attribution')
    copyright_text = str(copyright_text).strip().strip('\"')
    reflection = snippets[selected['reflectionSnippetId']]['text']
    reference = (passage_body.get('reference') or passage_body.get('human_reference')
                 or passage_body.get('display_reference') or selected['passageHint'])

    print('=' * 76)
    print(f'Grace in the Gap · {selected["durationSeconds"]}s pause · GLOO + YOUVERSION')
    print(f'Selector: {"live accepted" if accepted else "safe local fallback after rejection: " + rejection_reason}')
    print()
    print(f'"{text}"')
    print(f'{reference} · {version}')
    print()
    print(reflection)
    print(f'Attribution: {copyright_text}')
    print('Privacy: raw prompt is neither stored nor transmitted.')
    print('=' * 76)
    return {'glooLive': True, 'youVersionLive': True, 'selectorDegraded': degraded}

credentials = get_credentials()
if all(credentials.values()):
    live_proof = run_live_card(credentials)
    assert live_proof['glooLive'] and live_proof['youVersionLive']
else:
    print('LIVE CALL SKIPPED — add the three private Kaggle Secrets and Run All.')
    print('No result has been fabricated; every offline verification above remains complete.')


LIVE CALL SKIPPED — add the three private Kaggle Secrets and Run All.
No result has been fabricated; every offline verification above remains complete.


## Evidence map and conclusion

The notebook deliberately stops at the verifiable algorithmic and provider boundary. The complete product—including the async Claude Code hook, official spinner-tip integration, MCP server, API service, retry/timeout behavior, and 20-test suite—lives in the pinned TypeScript repository:

- [Weighted selector](https://github.com/Marc-Dvci/Grace-in-the-Gap/blob/d897c7f38c2f1a00a60879d1c2e028483d135c46/src/selection/local-selector.ts)
- [Privacy normalization](https://github.com/Marc-Dvci/Grace-in-the-Gap/blob/d897c7f38c2f1a00a60879d1c2e028483d135c46/src/privacy/normalize.ts)
- [Gloo structured selector](https://github.com/Marc-Dvci/Grace-in-the-Gap/blob/d897c7f38c2f1a00a60879d1c2e028483d135c46/src/providers/gloo.ts)
- [YouVersion passage and attribution adapter](https://github.com/Marc-Dvci/Grace-in-the-Gap/blob/d897c7f38c2f1a00a60879d1c2e028483d135c46/src/providers/youversion.ts)
- [Automated tests](https://github.com/Marc-Dvci/Grace-in-the-Gap/tree/d897c7f38c2f1a00a60879d1c2e028483d135c46/tests)

**Verified here:** exact catalog integrity, privacy-safe provider payload, production-parity selection across 108/108 scenarios, strict rejection of unsafe model output, and a credential-gated live API path that never substitutes fabricated Scripture.
